# DocuHelp: Intelligent Document Assistant  
### ITAI 2376 – Final Project  
**Student:** Jaret Sanchez  
**Environment:** Google Colab  
**Features:**  
- PDF ingestion  
- Text extraction  
- Chunking  
- Embeddings + FAISS retrieval  
- Topic classification (local ML model)  
- LangGraph agent for summarization  


# 1. Setup & Installation
This section installs all required libraries for DocuHelp and loads environment variables securely.


In [1]:
!pip install langchain langgraph faiss-cpu sentence-transformers pypdf python-dotenv openai transformers torch langchain-community langchain-huggingface langchain-openai

# 2. Load API Keys (Hidden)
Your API key will NOT be visible in the notebook or GitHub.

In [2]:
import os, getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


# 3. Upload PDF and Extract Text
Upload any PDF document. The agent will extract and process the text.

In [3]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_path)
pages = loader.load()

raw_text = "\n".join([p.page_content for p in pages])
raw_text[:1000]

Saving A_Brief_Introduction_To_AI.pdf to A_Brief_Introduction_To_AI (1).pdf


'A Brief Introduction to Artificial IntelligenceWhat is AI and how is it going to shape the futureBy Dibbyo Saha, Undergraduate Student, Computer Science,Ryerson University\nWhat is Artificial Intelligence?\nImage by Gerd Altmann from Pixabay\nGenerally speaking, Artificial Intelligence is a computing concept that helps amachinethinkandsolvecomplexproblemsaswehumansdowithourintelligence.For example, we performa task, make mistakes and learn fromour mistakes (Atleast thewiseonesofusdo!).Likewise,anAIorArtificialIntelligenceissupposedtoworkonaproblem, makesomemistakes insolvingtheproblemandlearnfromthe problems in a self-correcting manner as a part of its self-improvement. Or inother words, thinkof this likeplayingagameof chess.Everybadmoveyoumakereduces your chances of winning the game. So, every time you loseagainst yourfriend, you try remembering the moves you made which you shouldn’t have andapplythat knowledgeinyour next gameandsoon. Eventually, youget better andyour precision, or i

# 4. Text Splitting
Long documents are split into overlapping chunks for better retrieval.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = splitter.split_text(raw_text)
len(chunks)

21

# 5. Embeddings and Vector Store
We convert text chunks into embeddings and store them in FAISS for retrieval.

In [5]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize the SentenceTransformer model
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

# Wrap the SentenceTransformer model with HuggingFaceEmbeddings
embedder = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", model_kwargs={'device': 'cpu'})

docs = [Document(page_content=c) for c in chunks]

faiss_store = FAISS.from_documents(
    docs,
    embedding=embedder
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# 6. Topic Classification
A lightweight TF‑IDF + Logistic Regression classifier predicts the document's topic using the AG News dataset. This avoids Hugging Face authentication issues.


In [6]:
# Local AG News classifier (no Hugging Face required)
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# Download AG News dataset (public CSV)
!wget -q https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

# Load training data
train = pd.read_csv("train.csv", header=None)
X_train = train[1]
y_train = train[0] - 1  # labels 1–4 → 0–3

# Train TF-IDF + Logistic Regression classifier
vectorizer = TfidfVectorizer(stop_words="english", max_features=50000)
X_train_vec = vectorizer.fit_transform(X_train)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train_vec, y_train)

# Predict topic for your document
X_input = vectorizer.transform([raw_text[:1000]])
pred = clf.predict(X_input)[0]

topic_map = {
    0: "World News",
    1: "Sports",
    2: "Business",
    3: "Technology"
}

predicted_topic = topic_map[pred]
predicted_topic

'Technology'

# 7. LangGraph Agent
The agent retrieves relevant chunks and generates a summary using an LLM.

In [7]:
from typing import TypedDict, List

class AgentState(TypedDict):
    query: str
    retrieved_docs: List[str]
    summary: str

In [8]:
def retrieve_tool(state: AgentState):
    query = state["query"]
    results = faiss_store.similarity_search(query, k=4)
    state["retrieved_docs"] = [r.page_content for r in results]
    return state

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

def summarize_tool(state: AgentState):
    context = "\n\n".join(state["retrieved_docs"])
    prompt = f"Summarize the following document:\n\n{context}"
    response = llm.invoke(prompt)
    state["summary"] = response.content
    return state

In [10]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

graph.add_node("retrieve", retrieve_tool)
graph.add_node("summarize", summarize_tool)

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "summarize")
graph.add_edge("summarize", END)

agent = graph.compile()

# 8. Run the Agent
Ask the agent to summarize the document.

In [11]:
query = "Summarize the main ideas of this document."
result = agent.invoke({"query": query})
result["summary"]

'The document "A Brief Introduction to Artificial Intelligence" by Dibbyo Saha discusses the potential impact of AI on various sectors, particularly education and transportation. It highlights how AI can tailor educational experiences to meet individual needs, enhance commuting through technologies like self-driving cars and drones, and reshape the job market by potentially creating new roles despite concerns about automation displacing jobs. The document also illustrates current applications of AI through examples like smart personal assistants (e.g., Siri, Alexa) and recommendation algorithms used by platforms such as Netflix, which improve their accuracy as they gather more data from user interactions. Overall, the document emphasizes AI\'s transformative role in shaping the future.'

# 9. Final Output
Display the predicted topic and the generated summary.

In [12]:
print("Predicted Topic:", predicted_topic)
print("\nGenerated Summary:\n")
print(result["summary"])

Predicted Topic: Technology

Generated Summary:

The document "A Brief Introduction to Artificial Intelligence" by Dibbyo Saha discusses the potential impact of AI on various sectors, particularly education and transportation. It highlights how AI can tailor educational experiences to meet individual needs, enhance commuting through technologies like self-driving cars and drones, and reshape the job market by potentially creating new roles despite concerns about automation displacing jobs. The document also illustrates current applications of AI through examples like smart personal assistants (e.g., Siri, Alexa) and recommendation algorithms used by platforms such as Netflix, which improve their accuracy as they gather more data from user interactions. Overall, the document emphasizes AI's transformative role in shaping the future.


# Reflection
Working on this project taught me how many different pieces have to come together to make an AI system actually work. I went in thinking it would be pretty straightforward, but once I started connecting the PDF loader, the text splitter, the embeddings, FAISS, the classifier, and the LangGraph agent, I realized how much coordination is involved. The biggest challenge was definitely the Hugging Face issues. No matter what I tried, the models wouldn’t load in Colab, so I had to switch to a local classifier. It was frustrating at first, but it ended up being a good lesson in adapting when something breaks and finding another way to reach the same goal.

Building the LangGraph agent was actually one of the more interesting parts. I liked seeing how the workflow becomes cleaner when you break it into nodes instead of writing one giant block of code. It made the whole process feel more organized and I could actually see how the state moves through the graph. Overall, this project helped me understand retrieval‑augmented systems in a more hands‑on way. I got more comfortable with embeddings, vector stores, and agent design, and I also learned how to troubleshoot when things don’t go the way you expect.
